## 1. Schema


In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

project_root_text = str(PROJECT_ROOT)
if project_root_text in sys.path:
    sys.path.remove(project_root_text)
sys.path.insert(0, project_root_text)
importlib.invalidate_caches()

import safe_speed_schema
import safe_speed_data
import safe_speed_eda
import safe_speed_risk
import vru_evidence

safe_speed_schema = importlib.reload(safe_speed_schema)
safe_speed_data = importlib.reload(safe_speed_data)
safe_speed_eda = importlib.reload(safe_speed_eda)
safe_speed_risk = importlib.reload(safe_speed_risk)
vru_evidence = importlib.reload(vru_evidence)

schema_rows = safe_speed_schema.schema_rows
filter_valid_analysis_rows = safe_speed_data.filter_valid_analysis_rows
load_roads = safe_speed_data.load_roads
validate_schema = safe_speed_data.validate_schema
column_overview = safe_speed_eda.column_overview
duplicate_id_summary = safe_speed_eda.duplicate_id_summary
numeric_summary = safe_speed_eda.numeric_summary
speed_field_preview = safe_speed_eda.speed_field_preview
value_counts_table = safe_speed_eda.value_counts_table
DEFAULT_PRIORITY_WEIGHTS = safe_speed_risk.DEFAULT_PRIORITY_WEIGHTS
add_priority_score = safe_speed_risk.add_priority_score
add_speed_risk_metrics = safe_speed_risk.add_speed_risk_metrics
export_priority_segments = safe_speed_risk.export_priority_segments
summarize_speed_risk = safe_speed_risk.summarize_speed_risk
add_street_image_query_points = vru_evidence.add_street_image_query_points
add_vru_exposure_features = vru_evidence.add_vru_exposure_features
build_mapillary_vru_evidence_cache = vru_evidence.build_mapillary_vru_evidence_cache
build_segment_vru_check = vru_evidence.build_segment_vru_check
DEFAULT_VRU_EXPOSURE_WEIGHTS = vru_evidence.DEFAULT_VRU_EXPOSURE_WEIGHTS
setup_mapillary_client = vru_evidence.setup_mapillary_client
vru_status_summary = vru_evidence.vru_status_summary

display(pd.DataFrame(schema_rows()))


## 2. Load The Data


In [ ]:
GEOJSON_PATH = DATA_DIR / 'ADB_Innovation_Thailand.geojson'

features, roads_raw = load_roads(GEOJSON_PATH)
if roads_raw.empty:
    roads = pd.DataFrame()
    print(f'Place the challenge GeoJSON at: {GEOJSON_PATH}')
else:
    roads = roads_raw.copy()
    print(f'Loaded {len(roads_raw):,} road segments from {GEOJSON_PATH}')
    display(roads_raw.head())


## 3. EDA


In [ ]:
if roads_raw.empty:
    print('EDA will run after GEOJSON_PATH points to a real file.')
else:
    print(f'Rows / road segments: {len(roads_raw):,}')
    print(f'Columns: {len(roads_raw.columns):,}')
    print(f'Raw GeoJSON features: {len(features):,}')

    display(column_overview(roads_raw))

    for col in ['_geometry_type', 'RoadClass', 'LandUse', 'PercentileBand', 'AnalysisStatus']:
        table = value_counts_table(roads_raw, col)
        if not table.empty:
            print(f'Top values: {col}')
            display(table)

    display(numeric_summary(roads_raw))
    display(duplicate_id_summary(roads_raw))
    display(speed_field_preview(roads_raw))


## 4. Filter


In [ ]:
roads = filter_valid_analysis_rows(roads_raw)

if roads_raw.empty:
    print('Valid-row filter will run after the GeoJSON is loaded.')
elif 'AnalysisStatus' in roads_raw.columns:
    print(f'Filtered working dataset to AnalysisStatus == Valid: {len(roads):,} of {len(roads_raw):,} road segments')
else:
    print('AnalysisStatus column not found; continuing with all loaded road segments')

display(validate_schema(roads))
display(roads.head())


## 5. Compute Speed Gap


In [ ]:
roads_scored = add_speed_risk_metrics(roads) if not roads.empty else roads.copy()

risk_columns = [
    'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'SpeedLimit', 'MedianSpeed',
    'F85thPercentileSpeed', 'speed_gap_85th', 'PercentOverLimit', 'WeightedSample',
    'weighted_over_limit_pressure', 'over_limit_per_km', 'PercentileBand'
]

if roads_scored.empty:
    print('Risk metrics will populate after loading the GeoJSON.')
else:
    display(roads_scored[[c for c in risk_columns if c in roads_scored.columns]].head())
    display(summarize_speed_risk(roads_scored))


## 6. Prioritised Segment


In [ ]:
PRIORITY_WEIGHTS = DEFAULT_PRIORITY_WEIGHTS.copy()
# Edit these values to tune the prioritization model. The score is normalized by the active weight total.
PRIORITY_WEIGHTS.update({
    'weighted_over_limit_pressure': 0.45,
    'speed_gap_85th': 0.30,
    'PercentOverLimit': 0.15,
    'WeightedSample': 0.10,
})

prioritized = add_priority_score(roads_scored, weights=PRIORITY_WEIGHTS) if not roads_scored.empty else roads_scored.copy()

if prioritized.empty:
    print('Priority ranking will run after data is loaded.')
else:
    display(pd.DataFrame(PRIORITY_WEIGHTS.items(), columns=['factor', 'weight']))
    top_cols = [c for c in [
        'priority_score', 'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'SpeedLimit',
        'F85thPercentileSpeed', 'speed_gap_85th', 'PercentOverLimit', 'WeightedSample',
        'weighted_over_limit_pressure', 'PercentileBand', 'StreetImageLink'
    ] if c in prioritized.columns]
    display(prioritized.sort_values('priority_score', ascending=False)[top_cols].head(25))


## 7. Get Coordinates


In [ ]:
prioritized = add_street_image_query_points(prioritized) if not prioritized.empty else prioritized

if prioritized.empty:
    print('Street-image query points will be parsed after priority scoring runs.')
else:
    display(prioritized[['OBJECTID', 'StreetImageLink', 'query_lon', 'query_lat', 'street_image_point_count']].head())


## 8. VRU Evidencing Using Mapillary


In [ ]:
VRU_EVIDENCE_PATH = DATA_DIR / 'mapillary_vru_evidence.csv'
# Build or resume the Mapillary evidence cache. Existing OBJECTIDs are skipped when overwrite is False.
BUILD_MAPILLARY_CACHE = True
OVERWRITE_MAPILLARY_CACHE = False
# None targets all prioritized segments. Use an integer like 5 for a quick test.
MAPILLARY_SEGMENT_LIMIT = None
# None processes all remaining coordinate-eligible segments. Use an integer like 250 for smaller batches.
MAPILLARY_BATCH_SIZE = None
MAPILLARY_CACHE_TOP_N = len(prioritized) if MAPILLARY_SEGMENT_LIMIT is None else MAPILLARY_SEGMENT_LIMIT
MAPILLARY_SEARCH_RADIUS_M = 100
# Only segments with query_lon/query_lat can be checked through Mapillary.
MAPILLARY_SKIP_MISSING_COORDS = True
# Keep this low for full-dataset runs. Set to None only when you want every Mapillary image.
MAPILLARY_IMAGES_PER_SEGMENT = 1
# Thumbnail URLs add per-image calls; keep off for quick evidence runs.
MAPILLARY_INCLUDE_THUMBNAILS = False
# Segment-level Mapillary features/traffic signs add API calls; keep True for VRU/school/crossing evidence.
USE_MAPILLARY_API_FILTERS = True
# Per-image detections can be slow. Turn this on only for deeper evidence review.
USE_MAPILLARY_IMAGE_DETECTIONS = False
MAPILLARY_DETECTION_IMAGES_PER_SEGMENT = 1
# Parallel API calls. Keep this modest to avoid rate limits; try 4, 6, or 8.
MAPILLARY_MAX_WORKERS = 6
MAPILLARY_FLUSH_EVERY = 25
MAPILLARY_PROGRESS_EVERY = 25

mapillary_target = prioritized.sort_values('priority_score', ascending=False).head(MAPILLARY_CACHE_TOP_N) if 'priority_score' in prioritized.columns else prioritized.head(MAPILLARY_CACHE_TOP_N)
mapillary_coord_ready = mapillary_target['query_lon'].notna() & mapillary_target['query_lat'].notna()
print(f'Mapillary coordinate-eligible segments: {int(mapillary_coord_ready.sum())} of {len(mapillary_target)} target segment(s)')

mly = setup_mapillary_client(PROJECT_ROOT)
print(
    f'Mapillary run config: target_segments={MAPILLARY_CACHE_TOP_N}, '
    f'batch_size={MAPILLARY_BATCH_SIZE}, '
    f'images_per_segment={MAPILLARY_IMAGES_PER_SEGMENT}, '
    f'thumbnails={MAPILLARY_INCLUDE_THUMBNAILS}, '
    f'image_detections={USE_MAPILLARY_IMAGE_DETECTIONS}, '
    f'workers={MAPILLARY_MAX_WORKERS}'
)

if BUILD_MAPILLARY_CACHE:
    cache = build_mapillary_vru_evidence_cache(
        prioritized,
        evidence_path=VRU_EVIDENCE_PATH,
        mly=mly,
        top_n=MAPILLARY_CACHE_TOP_N,
        radius_m=MAPILLARY_SEARCH_RADIUS_M,
        max_images_per_segment=MAPILLARY_IMAGES_PER_SEGMENT,
        include_thumbnails=MAPILLARY_INCLUDE_THUMBNAILS,
        use_mapillary_api_filters=USE_MAPILLARY_API_FILTERS,
        use_image_detections=USE_MAPILLARY_IMAGE_DETECTIONS,
        max_detection_images_per_segment=MAPILLARY_DETECTION_IMAGES_PER_SEGMENT,
        overwrite=OVERWRITE_MAPILLARY_CACHE,
        batch_size=MAPILLARY_BATCH_SIZE,
        flush_every=MAPILLARY_FLUSH_EVERY,
        progress_every=MAPILLARY_PROGRESS_EVERY,
        skip_missing_coords=MAPILLARY_SKIP_MISSING_COORDS,
        max_workers=MAPILLARY_MAX_WORKERS,
    )
    cached_segments = cache['OBJECTID'].nunique() if not cache.empty and 'OBJECTID' in cache.columns else 0
    print(f'Updated Mapillary evidence cache: {VRU_EVIDENCE_PATH}')
    print(f'Cached coordinate-eligible segment(s): {cached_segments}')
    display(cache.head())
elif VRU_EVIDENCE_PATH.exists():
    print(f'Using existing Mapillary evidence cache: {VRU_EVIDENCE_PATH}')
    display(pd.read_csv(VRU_EVIDENCE_PATH).head())
else:
    print('Mapillary cache not built. Set BUILD_MAPILLARY_CACHE = True to create it from the API.')

vru_segment_check = build_segment_vru_check(prioritized, VRU_EVIDENCE_PATH)
if vru_segment_check.empty:
    print('VRU segment check will run after prioritized segments are available.')
else:
    review_cols = [c for c in [
        'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'priority_score',
        'query_lon', 'query_lat', 'mapillary_has_coverage', 'mapillary_image_count',
        'vru_presence_status', 'vru_signal_count', 'vru_check_action',
        'mapillary_images_cached', 'mapillary_api_value_count', 'mapillary_api_values', 'mapillary_api_errors',
    ] if c in vru_segment_check.columns]
    sort_col = 'priority_score' if 'priority_score' in vru_segment_check.columns else 'OBJECTID'
    display(vru_segment_check.sort_values(sort_col, ascending=False)[review_cols].head(50))
    display(vru_status_summary(vru_segment_check))



## 9. Risk Analysis


In [ ]:
VRU_EXPOSURE_WEIGHTS = DEFAULT_VRU_EXPOSURE_WEIGHTS.copy()
# Edit these values to tune the VRU exposure model.
# school_context and market_context fall back to school_or_market_context when separate labels are not available.
VRU_EXPOSURE_WEIGHTS.update({
    'is_urban_land_use': 0.20,
    'road_class_vru_weight': 0.12,
    'traffic_component': 0.13,
    'sidewalk_present': 0.08,
    'crossing_present': 0.12,
    'school_context': 0.07,
    'market_context': 0.07,
    'transit_stop_context': 0.06,
    'pedestrian_activity_visible': 0.10,
    'has_street_image_coords': 0.03,
    'has_mapillary_evidence': 0.02,
})

prioritized = (
    add_vru_exposure_features(prioritized, VRU_EVIDENCE_PATH, vru_weights=VRU_EXPOSURE_WEIGHTS)
    if not prioritized.empty
    else prioritized
)

if prioritized.empty:
    print('Risk analysis will run after data is loaded.')
else:
    display(pd.DataFrame(VRU_EXPOSURE_WEIGHTS.items(), columns=['factor', 'weight']))
    vru_cols = [c for c in [
        'safety_priority_score', 'priority_score', 'vru_exposure_score', 'OBJECTID', 'english_ro',
        'RoadClass', 'LandUse', 'is_urban_land_use', 'road_class_vru_weight', 'traffic_component',
        'sidewalk_present_factor', 'crossing_present_factor', 'school_context_factor', 'market_context_factor',
        'transit_stop_context_factor', 'pedestrian_activity_visible_factor',
        'manual_vru_indicator_score', 'has_manual_vru_evidence', 'has_mapillary_evidence',
        'mapillary_image_count', 'has_street_image_coords'
    ] if c in prioritized.columns]
    display(prioritized.sort_values('safety_priority_score', ascending=False)[vru_cols].head(100))

    out_path = export_priority_segments(prioritized, OUTPUT_DIR)
    if out_path is not None:
        print(f'Wrote {out_path}')


## 10. Visualization

Blank for now.


## 11. Policy Generation

Blank for now.
